# Génération Key Market Data

Ce notebook génère les Key Market Data en fusionnant :
- **CSV Performance** : Base principale avec toutes les métriques financières (~5500 tickers)
- **CSV S&P 500** : Enrichissement avec Weight et Price pour les 503 tickers S&P 500

**Résultat** : `all_market_data.json` avec toutes les entreprises


## 1. Imports et Configuration


In [1]:
import pandas as pd
import json
from pathlib import Path
from typing import Dict

# Chemins
BASE_DIR = Path('../../')
DATA_RAW = BASE_DIR / 'data' / 'raw'
DATA_GENERATED = BASE_DIR / 'data' / 'generated' / 'key_market_data'

# Créer dossier de sortie
DATA_GENERATED.mkdir(parents=True, exist_ok=True)

print(f"📁 Dossier de sortie: {DATA_GENERATED}")


📁 Dossier de sortie: ..\..\data\generated\key_market_data


## 2. Charger CSV Performance (Base principale)


In [2]:
# Charger CSV Performance (contient toutes les métriques financières)
performance_df = pd.read_csv(DATA_RAW / '2025-09-26_stocks-performance.csv')

# Normaliser colonne Symbol -> Ticker
if 'Symbol' in performance_df.columns:
    performance_df = performance_df.rename(columns={'Symbol': 'Ticker'})

print(f"✅ CSV Performance chargé: {len(performance_df)} entreprises")
print(f"   Colonnes: {list(performance_df.columns)}")
print(f"\n📊 Premières lignes:")
performance_df.head()


✅ CSV Performance chargé: 5508 entreprises
   Colonnes: ['Ticker', 'Company Name', 'Market Cap', 'Revenue', 'Op. Income', 'Net Income', 'EPS', 'FCF']

📊 Premières lignes:


,Ticker,Company Name,Market Cap,Revenue,Op. Income,Net Income,EPS,FCF
0,NVDA,NVIDIA Corporation,4.338392e+12,1.652180e+11,9.598100e+10,8.659700e+10,3.52,7.202300e+10
1,MSFT,Microsoft Corporation,3.801767e+12,2.817240e+11,1.285280e+11,1.018320e+11,13.64,7.161100e+10
2,AAPL,Apple Inc.,3.791126e+12,4.086250e+11,1.302140e+11,9.928000e+10,6.59,9.618400e+10
3,GOOG,Alphabet Inc.,2.989536e+12,3.713990e+11,1.213700e+11,1.155730e+11,9.37,6.672800e+10
4,GOOGL,Alphabet Inc.,2.981796e+12,3.713990e+11,1.213700e+11,1.155730e+11,9.38,6.672800e+10


## 3. Charger CSV S&P 500 (Enrichissement)


In [3]:
# Charger CSV S&P 500 (contient Weight et Price pour les 503 tickers)
composition_df = pd.read_csv(DATA_RAW / '2025-08-15_composition_sp500.csv')

# Normaliser colonne Symbol -> Ticker
if 'Symbol' in composition_df.columns:
    composition_df = composition_df.rename(columns={'Symbol': 'Ticker'})

# Convertir Weight et Price (format européen avec virgule)
if 'Weight' in composition_df.columns:
    composition_df['Weight'] = composition_df['Weight'].astype(str).str.replace(',', '.').astype(float)
if 'Price' in composition_df.columns:
    composition_df['Price'] = composition_df['Price'].astype(str).str.replace(',', '.').astype(float)

print(f"✅ CSV S&P 500 chargé: {len(composition_df)} entreprises")
print(f"   Colonnes: {list(composition_df.columns)}")
print(f"\n📊 Premières lignes:")
composition_df.head()


✅ CSV S&P 500 chargé: 503 entreprises
   Colonnes: ['#', 'Company', 'Ticker', 'Weight', 'Price']

📊 Premières lignes:


,#,Company,Ticker,Weight,Price
0,1,Nvidia,NVDA,0.0765,181.99
1,2,Microsoft,MSFT,0.0667,520.99
2,3,"Apple Inc,",AAPL,0.0597,233.28
3,4,Amazon,AMZN,0.0427,232.14
4,5,Meta Platforms,META,0.0339,783.78


## 4. Fusionner les données


In [ ]:
# Fusionner : Performance comme base, enrichir avec S&P 500
merged_df = pd.merge(
    performance_df,  # Base principale
    composition_df[['Ticker', 'Company', 'Weight', 'Price']],  # Enrichissement S&P 500
    on='Ticker',
    how='left',  # Garder tous les tickers de Performance, ajouter S&P 500 si disponible
    suffixes=('', '_sp500')
)

# Statistiques
sp500_count = merged_df['Weight'].notna().sum()
total_count = len(merged_df)

print(f"✅ Fusion terminée: {total_count} entreprises")
print(f"   - Tickers avec données S&P 500: {sp500_count}")
print(f"   - Tickers avec données performance seulement: {total_count - sp500_count}")

merged_df.head()


## 5. Créer structure JSON pour chaque entreprise


In [ ]:
def create_market_data_structure(row: pd.Series) -> Dict:
    """
    Crée une structure JSON enrichie pour les données de marché d'une entreprise
    """
    # Gérer company_name : priorité à Company (S&P 500), sinon Company Name (performance)
    company_name = None
    if pd.notna(row.get('Company')):
        company_name = str(row['Company']).strip()
    elif pd.notna(row.get('Company Name')):
        company_name = str(row['Company Name']).strip()
    
    # Déterminer si dans S&P 500
    is_sp500 = pd.notna(row.get('Weight'))
    
    # Structure de base
    data = {
        "ticker": str(row['Ticker']).upper() if pd.notna(row.get('Ticker')) else None,
        "company_name": company_name,
        "is_sp500": is_sp500,
        "sp500_weight": float(row['Weight']) if pd.notna(row.get('Weight')) else None,
        "current_price": float(row['Price']) if pd.notna(row.get('Price')) else None,
        "market_cap": float(row['Market Cap']) if pd.notna(row.get('Market Cap')) else None,
        "revenue": float(row['Revenue']) if pd.notna(row.get('Revenue')) else None,
        "operating_income": float(row['Op. Income']) if pd.notna(row.get('Op. Income')) else None,
        "net_income": float(row['Net Income']) if pd.notna(row.get('Net Income')) else None,
        "eps": float(row['EPS']) if pd.notna(row.get('EPS')) else None,
        "free_cash_flow": float(row['FCF']) if pd.notna(row.get('FCF')) else None,
    }
    
    # === CALCUL DES RATIOS FINANCIERS ===
    
    # 1. Profit Margin
    if data['revenue'] and data['net_income'] and data['revenue'] != 0:
        data['profit_margin'] = round(data['net_income'] / data['revenue'], 4)
    else:
        data['profit_margin'] = None
    
    # 2. Operating Margin
    if data['revenue'] and data['operating_income'] and data['revenue'] != 0:
        data['operating_margin'] = round(data['operating_income'] / data['revenue'], 4)
    else:
        data['operating_margin'] = None
    
    # 3. P/E Ratio (nécessite current_price)
    if data['current_price'] and data['eps'] and data['eps'] != 0:
        data['pe_ratio'] = round(data['current_price'] / data['eps'], 2)
    else:
        data['pe_ratio'] = None
    
    # 4. Price to Sales
    if data['market_cap'] and data['revenue'] and data['revenue'] != 0:
        data['price_to_sales'] = round(data['market_cap'] / data['revenue'], 2)
    else:
        data['price_to_sales'] = None
    
    # 5. FCF Margin
    if data['revenue'] and data['free_cash_flow'] and data['revenue'] != 0:
        data['fcf_margin'] = round(data['free_cash_flow'] / data['revenue'], 4)
    else:
        data['fcf_margin'] = None
    
    # 6. FCF Yield
    if data['market_cap'] and data['free_cash_flow'] and data['market_cap'] != 0:
        data['fcf_yield'] = round(data['free_cash_flow'] / data['market_cap'], 4)
    else:
        data['fcf_yield'] = None
    
    # 7. Earnings Yield
    if data['current_price'] and data['eps'] and data['current_price'] != 0:
        data['earnings_yield'] = round(data['eps'] / data['current_price'], 4)
    else:
        data['earnings_yield'] = None
    
    # 8. Market Cap to Earnings
    if data['market_cap'] and data['net_income'] and data['net_income'] != 0:
        data['market_cap_to_earnings'] = round(data['market_cap'] / data['net_income'], 2)
    else:
        data['market_cap_to_earnings'] = None
    
    # 9. Operating Efficiency
    if data['operating_income'] and data['net_income'] and data['net_income'] != 0:
        data['operating_efficiency'] = round(data['operating_income'] / data['net_income'], 2)
    else:
        data['operating_efficiency'] = None
    
    # 10. Cash Conversion
    if data['net_income'] and data['free_cash_flow'] and data['net_income'] != 0:
        data['cash_conversion'] = round(data['free_cash_flow'] / data['net_income'], 2)
    else:
        data['cash_conversion'] = None
    
    return data

# Test sur une ligne
if len(merged_df) > 0:
    example = create_market_data_structure(merged_df.iloc[0])
    print("✅ Exemple de structure générée:")
    print(json.dumps(example, indent=2, ensure_ascii=False))


## 6. Générer all_market_data.json


In [ ]:
# Créer dictionnaire pour toutes les entreprises
all_market_data = {}

for idx, row in merged_df.iterrows():
    ticker = str(row['Ticker']).upper() if pd.notna(row.get('Ticker')) else None
    if ticker:
        all_market_data[ticker] = create_market_data_structure(row)

print(f"✅ Structures créées: {len(all_market_data)} entreprises")

# Sauvegarder en JSON
output_file = DATA_GENERATED / "all_market_data.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_market_data, f, indent=2, ensure_ascii=False)

print(f"✅ Fichier sauvegardé: {output_file}")

# Statistiques finales
sp500_with_data = sum(1 for v in all_market_data.values() if v.get('is_sp500'))
print(f"\n📊 Statistiques:")
print(f"   - Total entreprises: {len(all_market_data)}")
print(f"   - Tickers S&P 500: {sp500_with_data}")
print(f"   - Tickers avec prix: {sum(1 for v in all_market_data.values() if v.get('current_price'))}")
print(f"   - Tickers avec company_name: {sum(1 for v in all_market_data.values() if v.get('company_name'))}")


## 7. Exemple : Afficher quelques entreprises


In [ ]:
# Exemples S&P 500
print("📊 Exemple S&P 500 (AAPL):")
if 'AAPL' in all_market_data:
    print(json.dumps(all_market_data['AAPL'], indent=2, ensure_ascii=False))

print("\n" + "="*80 + "\n")

# Exemple non-S&P 500
print("📊 Exemple non-S&P 500 (AA):")
if 'AA' in all_market_data:
    print(json.dumps(all_market_data['AA'], indent=2, ensure_ascii=False))


## 8. Sauvegarder aussi en CSV (optionnel)


In [ ]:
# Sauvegarder le DataFrame merged en CSV pour visualisation
csv_file = DATA_GENERATED / "merged_market_data.csv"
merged_df.to_csv(csv_file, index=False, encoding='utf-8')
print(f"✅ CSV sauvegardé: {csv_file}")
print(f"   Lignes: {len(merged_df)}")
print(f"   Colonnes: {len(merged_df.columns)}")
